# 자료 분석과 AI의 역할

작은 예제로 원리 확인 → 코드 실행 → 한 조건 변경 → 결과 해석

위에서 아래로 실행. 일반 CPU로 진행 가능. 날짜·평가 운영은 학기 운영안내에서 확인

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=5, suppress=True)

## 결측 처리에 따라 달라지는 평균

수위 자료는 `[1.2, 결측, 1.8] m`. 결측을 제외한 평균과 0으로 채운 평균을 각각 구하시오. 두 값의 차이를 설명하시오.

In [2]:
level = pd.Series([1.2, np.nan, 1.8], name='level_m')
print('결측 제외:', level.mean())
print('0으로 채움:', level.fillna(0).mean())
assert np.isclose(level.mean(), 1.5)
assert np.isclose(level.fillna(0).mean(), 1.0)

결측 제외: 1.5
0으로 채움: 1.0


결측은 관측값 0과 다른 의미. 먼저 단위·결측 표시·집계 규칙 확인.

## 공정표 검사

선행 작업이 끝나기 전에 시작한 작업을 찾으시오. 종료와 시작이 같은 시점이면 정상으로 처리하시오.

In [3]:
jobs = pd.DataFrame({'job':['A','B','C'], 'start':[1,1,4],
                     'end':[5,3,6], 'parent':[None,None,'A']})
def find_violations(table):
    by_job = table.set_index('job')
    rows = []
    for row in table.itertuples():
        if row.parent is None:
            continue
        if row.parent not in by_job.index or pd.isna(row.start):
            rows.append((row.job, '추가 확인 필요'))
        elif row.start < by_job.loc[row.parent, 'end']:
            rows.append((row.job, '선행 작업 종료 전 시작'))
    return rows
print(find_violations(jobs))
changed = jobs.copy()
changed.loc[changed.job == 'C', 'start'] = 5
assert find_violations(changed) == []

[('C', '선행 작업 종료 전 시작')]


## AI에 맡길 범위

입력 `jobs.csv`, 시각의 단위, 경계 허용 규칙, 누락 처리, 출력 `result.csv`를 함께 제시. 입력 자료 보존과 검사 예시 포함.

위 코드에 종료일 누락과 알 수 없는 선행 작업이 들어오면 어떻게 되는지 확인하시오. 부족한 처리를 설명하고 개선하시오.

원리·자료 해석은 학습의 중심. AI는 구현 초안·오류 설명을 돕는 도구. 실행 성공과 분석의 타당성은 별도로 확인